In [1]:
# =========================================
# [셀 1] Finnhub Earnings Surprise 수집 - 환경 설정
# =========================================
# 목적:
#   - Finnhub API로 S&P500 Earnings Surprise 수집
#   - fast_catalyst 데이터 확보
#
# 산출물:
#   - data/raw/finnhub/earnings_surprise.parquet
#
# 주의:
#   - 무료 티어: 60 calls/min → sleep 필요
#   - 체크포인트 저장으로 중단/재개 가능

import os
import time
import json
import requests
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime

# -----------------------------
# 프로젝트 루트 (SSOT)
# -----------------------------
QP2_ROOT = Path(r"C:\QP2")
DATA_DIR = QP2_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
META_DIR = DATA_DIR / "meta"

FINNHUB_DIR = RAW_DIR / "finnhub"
FINNHUB_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# API 설정
# -----------------------------
FINNHUB_API_KEY = "d5u2mfpr01qtjet1q8bgd5u2mfpr01qtjet1q8c0"
FINNHUB_BASE_URL = "https://finnhub.io/api/v1"

# Rate limit: 60/min → 1초에 1콜 안전하게
RATE_LIMIT_SLEEP = 1.1

# -----------------------------
# 경로
# -----------------------------
PATHS = {
    "sp500_universe": META_DIR / "sp500_universe.parquet",
    "earnings_out": FINNHUB_DIR / "earnings_surprise.parquet",
    "checkpoint": FINNHUB_DIR / "earnings_ckpt.parquet",
}

def _assert_exists(p: Path, name: str):
    if not p.exists():
        raise FileNotFoundError(f"[{name}] 파일이 없소: {p}")

print("✅ [셀1] Finnhub 환경 설정 완료")
print(f"- FINNHUB_DIR: {FINNHUB_DIR}")
print(f"- Rate limit sleep: {RATE_LIMIT_SLEEP}s")

✅ [셀1] Finnhub 환경 설정 완료
- FINNHUB_DIR: C:\QP2\data\raw\finnhub
- Rate limit sleep: 1.1s


In [2]:
# =========================================
# [셀 2] S&P500 티커 로드 + Finnhub API 함수
# =========================================
# 목적:
#   - S&P500 유니버스에서 티커 목록 추출
#   - Finnhub Earnings API 호출 함수 정의
#
# 산출물:
#   - tickers: S&P500 티커 리스트
#   - fetch_earnings(): API 호출 함수
#
# 주의:
#   - Finnhub은 ticker 형식이 Yahoo와 동일 (대부분)
#   - 일부 예외: BRK.B → BRK-B 등

# -----------------------------
# 1) S&P500 티커 로드
# -----------------------------
_assert_exists(PATHS["sp500_universe"], "sp500_universe")

sp500 = pd.read_parquet(PATHS["sp500_universe"])

# 티커 컬럼 확인 (ticker_yahoo 또는 ticker)
if "ticker_yahoo" in sp500.columns:
    tickers = sp500["ticker_yahoo"].dropna().unique().tolist()
elif "ticker" in sp500.columns:
    tickers = sp500["ticker"].dropna().unique().tolist()
else:
    raise RuntimeError(f"티커 컬럼을 찾을 수 없소: {sp500.columns.tolist()}")

tickers = sorted([str(t).strip() for t in tickers if t])
print(f"✅ S&P500 티커 로드: {len(tickers)}개")
print(f"   샘플: {tickers[:5]} ... {tickers[-5:]}")

# -----------------------------
# 2) Finnhub Earnings API 함수
# -----------------------------
def fetch_earnings(ticker: str, limit: int = 20) -> list:
    """
    Finnhub Earnings Surprise API 호출.
    
    Returns: list of dict
        [{"actual": 1.23, "estimate": 1.20, "period": "2023-12-31", 
          "surprise": 0.03, "surprisePercent": 2.5, "symbol": "AAPL"}, ...]
    
    실패 시 빈 리스트 반환.
    """
    url = f"{FINNHUB_BASE_URL}/stock/earnings"
    params = {
        "symbol": ticker,
        "limit": limit,
        "token": FINNHUB_API_KEY
    }
    
    try:
        r = requests.get(url, params=params, timeout=15)
        
        if r.status_code == 200:
            data = r.json()
            # Finnhub은 list로 반환
            if isinstance(data, list):
                return data
            else:
                return []
        
        elif r.status_code == 429:
            # Rate limit hit
            print(f"⚠️ Rate limit hit for {ticker}, sleeping 60s...")
            time.sleep(60)
            return fetch_earnings(ticker, limit)  # 재시도
        
        else:
            # 다른 에러
            return []
    
    except Exception as e:
        print(f"⚠️ {ticker} 요청 실패: {e}")
        return []

# -----------------------------
# 3) 테스트 호출 (AAPL 1개만)
# -----------------------------
print("\n--- AAPL 테스트 호출 ---")
test_data = fetch_earnings("AAPL", limit=4)

if test_data:
    print(f"✅ 테스트 성공: {len(test_data)}개 earnings")
    for item in test_data[:2]:
        print(f"   {item.get('period')}: actual={item.get('actual')}, est={item.get('estimate')}, surprise%={item.get('surprisePercent')}")
else:
    print("❌ 테스트 실패: 데이터 없음 또는 API 에러")

print("\n✅ [셀2] 완료")

✅ S&P500 티커 로드: 503개
   샘플: ['A', 'AAPL', 'ABBV', 'ABNB', 'ABT'] ... ['XYZ', 'YUM', 'ZBH', 'ZBRA', 'ZTS']

--- AAPL 테스트 호출 ---
✅ 테스트 성공: 4개 earnings
   2025-12-31: actual=2.84, est=2.7257, surprise%=4.1934
   2025-09-30: actual=1.85, est=1.8075, surprise%=2.3513

✅ [셀2] 완료


In [3]:
# =========================================
# [셀 3] S&P500 전체 Earnings Surprise 수집
# =========================================
# 목적:
#   - S&P500 전 종목 Earnings Surprise 수집
#   - 체크포인트 저장으로 중단/재개 가능
#
# 산출물:
#   - earnings_all: 전체 결과 DataFrame
#   - 체크포인트: finnhub/earnings_ckpt.parquet
#
# 주의:
#   - 예상 소요시간: 503종목 × 1.1초 ≈ 9~10분
#   - 중간에 끊겨도 체크포인트부터 재개됨

# -----------------------------
# 1) 체크포인트 로드 (있으면)
# -----------------------------
done_tickers = set()
earnings_list = []

if PATHS["checkpoint"].exists():
    ckpt = pd.read_parquet(PATHS["checkpoint"])
    if len(ckpt) > 0 and "symbol" in ckpt.columns:
        done_tickers = set(ckpt["symbol"].unique())
        earnings_list.append(ckpt)
        print(f"✅ 체크포인트 로드: {len(done_tickers)}개 티커 완료됨")

# 남은 티커
remaining = [t for t in tickers if t not in done_tickers]
print(f"📋 수집 대상: {len(remaining)}개 (전체 {len(tickers)}개 중)")

# -----------------------------
# 2) 수집 루프
# -----------------------------
CHECKPOINT_EVERY = 30  # 30개마다 저장

new_rows = []
failed = []

for i, ticker in enumerate(remaining, 1):
    data = fetch_earnings(ticker, limit=20)
    
    if data:
        for item in data:
            item["symbol"] = ticker  # 확실히 넣기
        new_rows.extend(data)
    else:
        failed.append(ticker)
    
    # 진행 상황
    if i % 10 == 0 or i == len(remaining):
        print(f"   [{i}/{len(remaining)}] {ticker} - 누적 {len(new_rows)} rows, 실패 {len(failed)}")
    
    # 체크포인트 저장
    if i % CHECKPOINT_EVERY == 0:
        tmp_df = pd.DataFrame(new_rows)
        if len(earnings_list) > 0:
            tmp_df = pd.concat([earnings_list[0], tmp_df], ignore_index=True)
        tmp_df.to_parquet(PATHS["checkpoint"], index=False)
        print(f"   💾 체크포인트 저장 ({len(tmp_df)} rows)")
    
    # Rate limit
    time.sleep(RATE_LIMIT_SLEEP)

# -----------------------------
# 3) 최종 결합 + 저장
# -----------------------------
if new_rows:
    new_df = pd.DataFrame(new_rows)
    if len(earnings_list) > 0:
        earnings_all = pd.concat([earnings_list[0], new_df], ignore_index=True)
    else:
        earnings_all = new_df
else:
    earnings_all = earnings_list[0] if earnings_list else pd.DataFrame()

# 중복 제거 (symbol + period 기준)
if len(earnings_all) > 0 and "symbol" in earnings_all.columns and "period" in earnings_all.columns:
    earnings_all = earnings_all.drop_duplicates(subset=["symbol", "period"], keep="last")

# 체크포인트 최종 저장
earnings_all.to_parquet(PATHS["checkpoint"], index=False)

print(f"\n✅ 수집 완료")
print(f"- 총 rows: {len(earnings_all)}")
print(f"- 티커 수: {earnings_all['symbol'].nunique() if 'symbol' in earnings_all.columns else 0}")
print(f"- 실패: {len(failed)}개 {failed[:10]}{'...' if len(failed) > 10 else ''}")

📋 수집 대상: 503개 (전체 503개 중)
   [10/503] ADM - 누적 40 rows, 실패 0
   [20/503] AKAM - 누적 80 rows, 실패 0
   [30/503] AMP - 누적 120 rows, 실패 0
   💾 체크포인트 저장 (120 rows)
   [40/503] APP - 누적 160 rows, 실패 0
   [50/503] AXP - 누적 200 rows, 실패 0
   [60/503] BG - 누적 240 rows, 실패 0
   💾 체크포인트 저장 (240 rows)
   [70/503] BRO - 누적 280 rows, 실패 0
   [80/503] CBOE - 누적 320 rows, 실패 0
   [90/503] CHRW - 누적 360 rows, 실패 0
   💾 체크포인트 저장 (360 rows)
   [100/503] CMS - 누적 400 rows, 실패 0
   [110/503] CPB - 누적 440 rows, 실패 0
   [120/503] CTAS - 누적 480 rows, 실패 0
   💾 체크포인트 저장 (480 rows)
   [130/503] DAY - 누적 520 rows, 실패 0
   [140/503] DIS - 누적 560 rows, 실패 0
   [150/503] DVA - 누적 600 rows, 실패 0
   💾 체크포인트 저장 (600 rows)
   [160/503] EL - 누적 640 rows, 실패 0
   [170/503] ES - 누적 680 rows, 실패 0
   [180/503] EXR - 누적 720 rows, 실패 0
   💾 체크포인트 저장 (720 rows)
   [190/503] FIS - 누적 760 rows, 실패 0
   [200/503] GD - 누적 800 rows, 실패 0
   [210/503] GM - 누적 840 rows, 실패 0
   💾 체크포인트 저장 (840 rows)
   [220/503] HAS - 누적 880 rows, 실패

In [ ]:
# =========================================
# [셀 4] Earnings Surprise 정리 + 저장
# =========================================
# 문제 발생 ㅋㅋ 무료분이라 1년 반 바께 데이터 없엉.. 아마 단기 숔은 반영 포함 못할듯. 0으로 처리해야,..,
# 목적:
#   - 원시 데이터 정리 (컬럼명, 타입)
#   - surprise_pct 확인/계산
#   - 최종 parquet 저장
#
# 산출물:
#   - data/raw/finnhub/earnings_surprise.parquet
#
# 주의:
#   - period = 실적 발표 기준 분기말 (fiscal period end)
#   - surprise_pct = (actual - estimate) / |estimate| * 100

# -----------------------------
# 1) 데이터 로드 (체크포인트에서)
# -----------------------------
earnings = pd.read_parquet(PATHS["checkpoint"]).copy()

print("=== 원시 데이터 확인 ===")
print(f"shape: {earnings.shape}")
print(f"columns: {earnings.columns.tolist()}")
print(earnings.head(3))

# -----------------------------
# 2) 컬럼 정리
# -----------------------------
# Finnhub 반환 컬럼: actual, estimate, period, surprise, surprisePercent, symbol
# 우리가 필요한 것: symbol(ticker), period(date), actual, estimate, surprise_pct

col_map = {
    "symbol": "ticker",
    "period": "period",
    "actual": "eps_actual",
    "estimate": "eps_estimate",
    "surprise": "eps_surprise",
    "surprisePercent": "surprise_pct"
}

# 존재하는 컬럼만 rename
existing_cols = {k: v for k, v in col_map.items() if k in earnings.columns}
earnings = earnings.rename(columns=existing_cols)

# -----------------------------
# 3) 타입 변환
# -----------------------------
earnings["period"] = pd.to_datetime(earnings["period"], errors="coerce")
earnings["eps_actual"] = pd.to_numeric(earnings["eps_actual"], errors="coerce")
earnings["eps_estimate"] = pd.to_numeric(earnings["eps_estimate"], errors="coerce")

# surprise_pct가 없거나 null이면 직접 계산
if "surprise_pct" not in earnings.columns:
    earnings["surprise_pct"] = np.nan

mask_null = earnings["surprise_pct"].isna()
mask_est = earnings["eps_estimate"].notna() & (earnings["eps_estimate"].abs() > 0.001)

earnings.loc[mask_null & mask_est, "surprise_pct"] = (
    (earnings.loc[mask_null & mask_est, "eps_actual"] - 
     earnings.loc[mask_null & mask_est, "eps_estimate"]) / 
    earnings.loc[mask_null & mask_est, "eps_estimate"].abs() * 100
)

# -----------------------------
# 4) 정리 + 정렬
# -----------------------------
earnings = earnings.dropna(subset=["ticker", "period"])
earnings = earnings.sort_values(["ticker", "period"]).reset_index(drop=True)

# 필요한 컬럼만
keep_cols = ["ticker", "period", "eps_actual", "eps_estimate", "eps_surprise", "surprise_pct"]
keep_cols = [c for c in keep_cols if c in earnings.columns]
earnings = earnings[keep_cols]

print("\n=== 정리 후 ===")
print(f"shape: {earnings.shape}")
print(f"티커 수: {earnings['ticker'].nunique()}")
print(f"기간: {earnings['period'].min().date()} ~ {earnings['period'].max().date()}")
print(f"\nsurprise_pct 분포:")
print(earnings["surprise_pct"].describe().round(2))

# -----------------------------
# 5) 저장
# -----------------------------
earnings.to_parquet(PATHS["earnings_out"], index=False)
print(f"\n✅ 저장 완료: {PATHS['earnings_out']}")

# 샘플 출력
print("\n=== 샘플 (AAPL) ===")
display(earnings[earnings["ticker"] == "AAPL"].head(5))

=== 원시 데이터 확인 ===
shape: (2009, 8)
columns: ['actual', 'estimate', 'period', 'quarter', 'surprise', 'surprisePercent', 'symbol', 'year']
   actual  estimate      period  quarter  surprise  surprisePercent symbol  \
0    1.59    1.6158  2025-12-31        4   -0.0258          -1.5967      A   
1    1.37    1.3928  2025-09-30        3   -0.0228          -1.6370      A   
2    1.31    1.2899  2025-06-30        2    0.0201           1.5583      A   

   year  
0  2025  
1  2025  
2  2025  

=== 정리 후 ===
shape: (2009, 6)
티커 수: 503
기간: 2024-09-30 ~ 2025-12-31

surprise_pct 분포:
count    2005.00
mean       12.24
std       131.20
min     -1469.86
25%        -0.55
50%         3.06
75%         9.90
max      4376.74
Name: surprise_pct, dtype: float64

✅ 저장 완료: C:\QP2\data\raw\finnhub\earnings_surprise.parquet

=== 샘플 (AAPL) ===


,ticker,period,eps_actual,eps_estimate,eps_surprise,surprise_pct
4,AAPL,2025-03-31,1.65,1.6596,-0.0096,-0.5785
5,AAPL,2025-06-30,1.57,1.4626,0.1074,7.3431
6,AAPL,2025-09-30,1.85,1.8075,0.0425,2.3513
7,AAPL,2025-12-31,2.84,2.7257,0.1143,4.1934


In [5]:
# =========================================
# [셀 5] Fast Catalyst용 변환 + 검증
# =========================================
# 목적:
#   - Earnings Surprise → 월별 fast_catalyst 변환
#   - 분기 실적 → 해당 분기 이후 월에 매핑
#   - 02_Factor_04에서 로드할 수 있는 형태로 저장
#
# 산출물:
#   - data/interim/fast_catalyst.parquet
#
# 주의:
#   - 실적 발표일(announcement date)이 없으므로
#   - period(분기말) + 45일 = 시장 반영 가능 시점으로 가정
#   - 보수적 랙 적용

# -----------------------------
# 1) 데이터 로드
# -----------------------------
earnings = pd.read_parquet(PATHS["earnings_out"]).copy()

print(f"✅ Earnings 로드: {len(earnings)} rows, {earnings['ticker'].nunique()} tickers")
print(f"   기간: {earnings['period'].min().date()} ~ {earnings['period'].max().date()}")

# -----------------------------
# 2) Effective Date 계산 (period + 45일)
# -----------------------------
# 분기 실적은 보통 분기말 후 30~45일에 발표
# 보수적으로 45일 잡음 (look-ahead 방지)
EARNINGS_LAG_DAYS = 45

earnings["effective_date"] = earnings["period"] + pd.Timedelta(days=EARNINGS_LAG_DAYS)

# -----------------------------
# 3) 극단치 처리
# -----------------------------
# surprise_pct 극단치 winsorize (1%, 99%)
def winsorize(s, p_low=0.01, p_high=0.99):
    lo = s.quantile(p_low)
    hi = s.quantile(p_high)
    return s.clip(lower=lo, upper=hi)

earnings["surprise_pct_w"] = winsorize(earnings["surprise_pct"], 0.01, 0.99)

print(f"\n=== surprise_pct winsorized ===")
print(earnings["surprise_pct_w"].describe().round(2))

# -----------------------------
# 4) 티커별 가장 최근 surprise만 사용 (단순화)
# -----------------------------
# 여러 분기가 있으면 가장 최근 것만
earnings_latest = (
    earnings
    .sort_values(["ticker", "period"])
    .groupby("ticker")
    .tail(1)
    .reset_index(drop=True)
)

print(f"\n✅ 티커당 최근 1개: {len(earnings_latest)} rows")

# -----------------------------
# 5) 월별 리밸런스 매핑용 테이블 생성
# -----------------------------
# 02_Factor_04에서 merge_asof로 붙일 수 있게
# (ticker, effective_date, fast_cat)

fast_cat = earnings_latest[["ticker", "effective_date", "surprise_pct_w"]].copy()
fast_cat = fast_cat.rename(columns={"surprise_pct_w": "fast_cat"})
fast_cat = fast_cat.dropna(subset=["fast_cat"])

# effective_date 타입 통일
fast_cat["effective_date"] = pd.to_datetime(fast_cat["effective_date"]).astype("datetime64[ns]")

print(f"\n=== fast_cat 최종 ===")
print(f"shape: {fast_cat.shape}")
print(f"기간: {fast_cat['effective_date'].min().date()} ~ {fast_cat['effective_date'].max().date()}")
print(fast_cat.head())

# -----------------------------
# 6) 저장
# -----------------------------
OUT_PATH = DATA_DIR / "interim" / "fast_catalyst.parquet"
fast_cat.to_parquet(OUT_PATH, index=False)

print(f"\n✅ 저장 완료: {OUT_PATH}")

✅ Earnings 로드: 2009 rows, 503 tickers
   기간: 2024-09-30 ~ 2025-12-31

=== surprise_pct winsorized ===
count    2005.00
mean        7.57
std        26.22
min       -72.64
25%        -0.55
50%         3.06
75%         9.90
max       159.56
Name: surprise_pct_w, dtype: float64

✅ 티커당 최근 1개: 503 rows

=== fast_cat 최종 ===
shape: (502, 3)
기간: 2025-11-14 ~ 2026-02-14
  ticker effective_date  fast_cat
0      A     2026-02-14   -1.5967
1   AAPL     2026-02-14    4.1934
2   ABBV     2025-11-14    2.7454
3   ABNB     2025-11-14   -7.4656
4    ABT     2026-02-14   -0.7477

✅ 저장 완료: C:\QP2\data\interim\fast_catalyst.parquet


In [6]:
# =========================================
# [셀 6] 00_DataLoader_Finnhub_Earnings.ipynb 요약
# =========================================

# =============================================================================
# [파이프라인 요약]
# =============================================================================
# 목적: Finnhub API로 S&P500 Earnings Surprise 수집 → fast_catalyst 생성
#
# 산출물:
#   - data/raw/finnhub/earnings_surprise.parquet  (원시 데이터)
#   - data/interim/fast_catalyst.parquet          (팩터용 정제 데이터)
#
# =============================================================================
# [데이터 현황]
# =============================================================================
# - 티커 수: 503개 (S&P500 전체)
# - 기간: 2024-09-30 ~ 2025-12-31 (약 1년 반)
#   ⚠️ Finnhub 무료 티어 한계: 최근 4~5분기만 제공
#   ⚠️ 장기 백테스트에는 부적합, 최근 구간 검증용으로만 유효
#
# - fast_cat 분포 (winsorized):
#   mean=7.57%, std=26.22%, min=-72.64%, max=159.56%
#
# =============================================================================
# [사용법 - 02_Factor_04_A_integrated.ipynb에서]
# =============================================================================
# fast_cat = pd.read_parquet(DATA_DIR / "interim" / "fast_catalyst.parquet")
# 
# # 월말 리밸런스에 merge_asof로 매핑:
# # - left_on="date" (리밸런스 월말)
# # - right_on="effective_date"
# # - direction="backward"
#
# =============================================================================
# [한계 및 개선 방향]
# =============================================================================
# 1. 데이터 기간 부족 (1.5년)
#    → 유료 API (Refinitiv, Bloomberg) 또는 Polygon.io 고려
#
# 2. 발표일(announcement date) 없음
#    → 현재는 period + 45일로 보수적 추정
#    → SEC EDGAR 8-K 파싱하면 정확한 발표일 확보 가능
#
# 3. 분기별 1개만 사용
#    → 연속 서프라이즈 (2분기 연속 beat 등) 시그널 추가 가능
#
# =============================================================================
# END OF NOTEBOOK
# =============================================================================

print("✅ [셀6] 00_DataLoader_Finnhub_Earnings.ipynb 완료")
print("")
print("=== 핵심 요약 ===")
print("- 산출물: data/interim/fast_catalyst.parquet")
print("- 티커: 503개")
print("- 기간: 2024-09-30 ~ 2025-12-31 (⚠️ 약 1.5년, 무료 API 한계)")
print("")
print("다음: 02_Factor_04_A_integrated.ipynb에서 fast_catalyst 로드하여 통합")

✅ [셀6] 00_DataLoader_Finnhub_Earnings.ipynb 완료

=== 핵심 요약 ===
- 산출물: data/interim/fast_catalyst.parquet
- 티커: 503개
- 기간: 2024-09-30 ~ 2025-12-31 (⚠️ 약 1.5년, 무료 API 한계)

다음: 02_Factor_04_A_integrated.ipynb에서 fast_catalyst 로드하여 통합
